In [23]:
import re
import os
import gzip
import joblib
import warnings
import numpy as np
import pandas as pd
from scipy import sparse
from rapidfuzz import process, fuzz
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, LabelBinarizer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

df=pd.read_csv("../data/interim/cars_cleaned.csv")

In [24]:
df

,name,city,price,model,kilometer,fuel_type,engine,transmission,brand
0,Honda Civic 2021 1.5 RS Turbo,abbottabad,6700000.0,2021,88000,petrol,1500,automatic,honda
1,Honda Civic 2019 Oriel 1.8 i-VTEC CVT,peshawar,6150000.0,2019,68425,petrol,1800,automatic,honda
2,Toyota Hilux 2012 D-4D Automatic,peshawar,6550000.0,2012,132000,diesel,3000,automatic,toyota
3,Toyota Prius Alpha 2018 S Touring Selection G...,peshawar,9850000.0,2018,35000,hybrid,1800,automatic,toyota
4,Toyota Corolla Fielder 2017 Hybrid G WB,peshawar,6200000.0,2017,165044,hybrid,1500,automatic,toyota
...,...,...,...,...,...,...,...,...,...
4669,Suzuki Cultus 2007 VXR (CNG),mardan,1190000.0,2007,80000,petrol,1000,manual,suzuki
4670,Suzuki Alto 1998,peshawar,1270000.0,1998,120000,petrol,660,automatic,suzuki
4671,Suzuki Mehran 2004 VX,charsadda,550000.0,2004,100000,petrol,800,manual,suzuki
4672,Suzuki Alto 2011 VXR,peshawar,1499000.0,2011,95000,petrol,1000,manual,suzuki


In [25]:
ARTIFACT_DIR = "../data/processed"

PIPELINE_PATH = f"{ARTIFACT_DIR}/pipeline.joblib"
MATRIX_PATH   = f"{ARTIFACT_DIR}/feature_matrix.npz"
DATA_PATH     = f"{ARTIFACT_DIR}/cars_dataframe.parquet"

# Experiment Encoding

In [26]:
# =========================================================
# CUSTOM TRANSFORMERS
# =========================================================

class DropColumns(BaseEstimator, TransformerMixin):

    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.columns, errors="ignore")


class MultiLabelBinarizer(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.encoders_ = {}
        self.feature_names_ = []

    def fit(self, X, y=None):

        X = pd.DataFrame(X).reset_index(drop=True)

        for col in X.columns:

            lb = LabelBinarizer()
            lb.fit(X[col].astype(str))

            self.encoders_[col] = lb

            if len(lb.classes_) == 2:
                self.feature_names_.append(str(col))
            else:
                self.feature_names_.extend(
                    [f"{col}_{c}" for c in lb.classes_]
                )

        return self

    def transform(self, X):

        X = pd.DataFrame(X).reset_index(drop=True)

        parts = []

        for col in X.columns:
            lb = self.encoders_[col]
            transformed = lb.transform(X[col].astype(str))
            parts.append(transformed)

        return np.hstack(parts)

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_


class FrequencyEncoder(BaseEstimator, TransformerMixin):

    def __init__(self, cols):
        self.cols = cols
        self.freq_maps_ = {}
        self.min_freqs_ = {}

    def fit(self, X, y=None):

        X = pd.DataFrame(X)

        for col in X.columns:

            freq = (
                X[col]
                .astype(str)
                .value_counts(normalize=True)
                .to_dict()
            )

            self.freq_maps_[col] = freq
            self.min_freqs_[col] = min(freq.values())

        return self

    def transform(self, X):

        X = pd.DataFrame(X).copy()

        for col in X.columns:

            min_freq = self.min_freqs_.get(col, 0)

            X[col] = (
                X[col]
                .astype(str)
                .map(self.freq_maps_[col])
                .fillna(min_freq)
            )

        return X.values.astype(float)

    def get_feature_names_out(self, input_features=None):
        return list(self.cols)




# =========================================================
# COLUMN DEFINITIONS
# =========================================================

DROP_COLS = ['name']

NUMERIC_COLS = [
    'price',
    'kilometer',
    'engine',
    'model'
]

BINARIZE_COLS = [
    'fuel_type',
    'transmission'
]

FREQ_ENC_COLS = [
    'brand',
    'city'
]

ALL_FEATURE_COLS = (
    NUMERIC_COLS +
    BINARIZE_COLS +
    FREQ_ENC_COLS
)

# =========================================================
# BUILD PIPELINE
# =========================================================

def build_pipeline():

    numeric_pipe = Pipeline([
        ("scaler", StandardScaler())
    ])

    binarize_pipe = Pipeline([
        ("binarizer", MultiLabelBinarizer())
    ])

    freq_pipe = Pipeline([
        ("freq", FrequencyEncoder(cols=FREQ_ENC_COLS))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipe, NUMERIC_COLS),
        ("binarize", binarize_pipe, BINARIZE_COLS),
        ("freq", freq_pipe, FREQ_ENC_COLS),
    ])

    pipeline = Pipeline([
        ("drop_cols", DropColumns(DROP_COLS)),
        ("preprocessor", preprocessor)
    ])

    return pipeline

# Best Encoding

In [27]:
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.feature_extraction.text import TfidfVectorizer


# =========================================================
# CUSTOM TRANSFORMERS
# =========================================================

class DropColumns(BaseEstimator, TransformerMixin):

    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.columns, errors="ignore")


# =========================================================
# COLUMN DEFINITIONS
# =========================================================

DROP_COLS = ['name']

NUMERIC_COLS = [
    'price',
    'kilometer',
    'engine',
    'model'
]

# EXACT CATEGORIES
ONEHOT_COLS = [
    'fuel_type',
    'transmission'
]

# SEMANTIC CATEGORIES
TEXT_SIMILARITY_COLS = [
    'brand',
    'city'
]

ALL_FEATURE_COLS = (
    NUMERIC_COLS +
    ONEHOT_COLS +
    TEXT_SIMILARITY_COLS
)


# =========================================================
# TF-IDF COLUMN WRAPPER
# =========================================================

class TextColumnSelector(
    BaseEstimator,
    TransformerMixin
):

    def fit(self, X, y=None):
        return self

    def transform(self, X):

        if isinstance(X, pd.DataFrame):
            return X.iloc[:, 0].astype(str)

        return pd.Series(X.ravel()).astype(str)


# =========================================================
# BUILD PIPELINE
# =========================================================

def build_pipeline():

    # -----------------------------------------------------
    # NUMERIC FEATURES
    # -----------------------------------------------------

    numeric_pipe = Pipeline([
        ("scaler", StandardScaler())
    ])

    # -----------------------------------------------------
    # EXACT CATEGORICAL FEATURES
    # -----------------------------------------------------

    onehot_pipe = Pipeline([
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ])

    # -----------------------------------------------------
    # BRAND TF-IDF
    # -----------------------------------------------------

    brand_pipe = Pipeline([

        (
            "selector",
            TextColumnSelector()
        ),

        (
            "tfidf",
            TfidfVectorizer(

                analyzer="char_wb",

                ngram_range=(2, 4),

                lowercase=True
            )
        )
    ])

    # -----------------------------------------------------
    # CITY TF-IDF
    # -----------------------------------------------------

    city_pipe = Pipeline([

        (
            "selector",
            TextColumnSelector()
        ),

        (
            "tfidf",
            TfidfVectorizer(

                analyzer="char_wb",

                ngram_range=(2, 4),

                lowercase=True
            )
        )
    ])

    # -----------------------------------------------------
    # COLUMN TRANSFORMER
    # -----------------------------------------------------

    preprocessor = ColumnTransformer(

        transformers=[

            (
                "numeric",
                numeric_pipe,
                NUMERIC_COLS
            ),

            (
                "onehot",
                onehot_pipe,
                ONEHOT_COLS
            ),

            (
                "brand_text",
                brand_pipe,
                ["brand"]
            ),

            (
                "city_text",
                city_pipe,
                ["city"]
            )
        ],

        transformer_weights={

            # lower importance
            "numeric": 1.0,

            # medium importance
            "onehot": 2.0,

            # important semantic similarity
            "brand_text": 5.0,

            # medium semantic similarity
            "city_text": 2.0
        }
    )

    # -----------------------------------------------------
    # FINAL PIPELINE
    # -----------------------------------------------------

    pipeline = Pipeline([

        (
            "drop_cols",
            DropColumns(DROP_COLS)
        ),

        (
            "preprocessor",
            preprocessor
        )
    ])

    return pipeline

In [28]:

# =========================================================
# STAGE 1 — OFFLINE TRAINING WITH DASK
# =========================================================

def train_and_save_pipeline(
    filepath,
    file_type="csv",
    blocksize="64MB"
):
    """
    RUN THIS ONLY ONCE

    This:
        1. Loads huge data with Dask
        2. Fits sklearn pipeline
        3. Generates feature matrix
        4. Compresses + saves matrix
        5. Saves fitted pipeline
        6. Saves dataframe
    """

    print("\n⚡ LOADING DATA WITH DASK")

    import dask.dataframe as dd
    from dask.diagnostics import ProgressBar

    loaders = {
        "csv": lambda: dd.read_csv(filepath, blocksize=blocksize),
        "parquet": lambda: dd.read_parquet(filepath),
        "json": lambda: dd.read_json(filepath)
    }

    if file_type not in loaders:
        raise ValueError("Unsupported file type")

    df_dask = loaders[file_type]()

    print(f"Partitions: {df_dask.npartitions}")

    required_cols = DROP_COLS + ALL_FEATURE_COLS

    print("\n⚙️ COMPUTING REQUIRED COLUMNS")

    with ProgressBar():
        df = df_dask[required_cols].compute()

    print(f"\n✅ Rows Loaded: {len(df):,}")

    # -----------------------------------------------------
    # FIT PIPELINE
    # -----------------------------------------------------

    print("\n⚙️ FITTING PIPELINE")

    pipeline = build_pipeline()

    X = pipeline.fit_transform(df)
    print(X.shape)
    print("✅ Pipeline Fitted")

    # -----------------------------------------------------
    # CONVERT TO SPARSE MATRIX
    # -----------------------------------------------------

    print("\n⚙️ CONVERTING TO SPARSE MATRIX")

    X_sparse = sparse.csr_matrix(X)
   
    print("✅ Sparse Matrix Created")

    # -----------------------------------------------------
    # SAVE EVERYTHING
    # -----------------------------------------------------

    os.makedirs(ARTIFACT_DIR, exist_ok=True)

    print("\n💾 SAVING PIPELINE")

    joblib.dump(
        pipeline,
        PIPELINE_PATH,
        compress=("gzip", 3)
    )

    print("✅ Pipeline Saved")

    print("\n💾 SAVING COMPRESSED MATRIX")

    sparse.save_npz(
        MATRIX_PATH,
        X_sparse,
        compressed=True
    )

    print("✅ Matrix Saved")

    print("\n💾 SAVING DATAFRAME")

    df.to_parquet(DATA_PATH, index=False)

    print("✅ DataFrame Saved")

    print("\n🎉 TRAINING COMPLETE")

train_and_save_pipeline("../data/interim/cars_cleaned.csv")    


⚡ LOADING DATA WITH DASK
Partitions: 1

⚙️ COMPUTING REQUIRED COLUMNS
[########################################] | 100% Completed | 110.41 ms

✅ Rows Loaded: 4,674

⚙️ FITTING PIPELINE
(4674, 1117)
✅ Pipeline Fitted

⚙️ CONVERTING TO SPARSE MATRIX
✅ Sparse Matrix Created

💾 SAVING PIPELINE
✅ Pipeline Saved

💾 SAVING COMPRESSED MATRIX
✅ Matrix Saved

💾 SAVING DATAFRAME
✅ DataFrame Saved

🎉 TRAINING COMPLETE


In [29]:
# =========================================================
# LOAD ARTIFACTS
# =========================================================

def load_artifacts():

    print("\n📦 LOADING SAVED ARTIFACTS")

    pipeline = joblib.load(PIPELINE_PATH)

    X_sparse = sparse.load_npz(MATRIX_PATH)

    df = pd.read_parquet(DATA_PATH)

    print("✅ Artifacts Loaded")

    return pipeline, X_sparse, df


# =========================================================
# FUZZY MATCH
# =========================================================
from rapidfuzz import process
from rapidfuzz import fuzz


# =========================================================
# NORMALIZER
# =========================================================

def normalize_text(text):

    text = str(text).lower()

    # remove special chars
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# =========================================================
# BUILD SEARCH COLUMN
# =========================================================

def build_search_text(df):

    """
    Create ONE optimized searchable column.

    Run ONCE during preprocessing.
    """

    df = df.copy()

    df["search_text"] = (

        df["name"].astype(str) + " " +

        df["brand"].astype(str) + " " +

        df["fuel_type"].astype(str) + " " +

        df["transmission"].astype(str) + " " +

        df["model"].astype(str)
    )

    df["search_text"] = (
        df["search_text"]
        .apply(normalize_text)
    )

    return df


# =========================================================
# FAST FUZZY MATCH
# =========================================================

def fuzzy_match_car(
    user_input,
    df,
    threshold=60
):

    # normalize query
    query = normalize_text(user_input)
    df=build_search_text(df)
    # precomputed search texts
    choices = df["search_text"].tolist()

    # -----------------------------------------------------
    # TOKEN-BASED MATCHING
    # -----------------------------------------------------

    """
    token_set_ratio handles:

    "honda turbo"
    vs
    "Honda Civic 1.5 RS Turbo"

    VERY well.
    """

    result = process.extractOne(

        query,

        choices,

        scorer=fuzz.token_set_ratio,

        score_cutoff=threshold
    )

    # -----------------------------------------------------
    # NO MATCH
    # -----------------------------------------------------

    if result is None:
        return None, 0

    matched_text, score, idx = result

    matched_row = df.iloc[[idx]]

    print("\n🔍 MATCHED:")
    print(matched_row["name"].values[0])

    print(f"Score: {round(score,2)}")

    return matched_row, score





# =========================================================
# ONLINE INFERENCE
# =========================================================

def recommend_cars(
    user_input,
    top_n=5
):
    """
    FAST ONLINE INFERENCE

    Flow:
        1. Load compressed matrix
        2. Load pipeline
        3. Transform user row
        4. Similarity search
    """

    # -----------------------------------------------------
    # LOAD SAVED OBJECTS
    # -----------------------------------------------------

    pipeline, X_sparse, df = load_artifacts()

    # -----------------------------------------------------
    # FIND MATCH
    # -----------------------------------------------------

    matched_row, score = fuzzy_match_car(
        user_input,
        df
    )

    # -----------------------------------------------------
    # FALLBACK
    # -----------------------------------------------------
    print(matched_row)
    if matched_row is None:

        print("\n↩️ USING FALLBACK PARSER")
        return None

    # -----------------------------------------------------
    # TRANSFORM SINGLE USER INPUT
    # -----------------------------------------------------

    user_vector = pipeline.transform(matched_row)

    user_vector = sparse.csr_matrix(user_vector)

    # -----------------------------------------------------
    # COSINE SIMILARITY
    # -----------------------------------------------------

    similarities = cosine_similarity(
        user_vector,
        X_sparse
    )[0]

    # -----------------------------------------------------
    # GET TOP RESULTS
    # -----------------------------------------------------

    indices = np.argsort(similarities)[::-1]

    matched_name = matched_row["name"].values[0]

    results = []

    for idx in indices:

        if df.iloc[idx]["name"] == matched_name:
            continue

        results.append(idx)

        if len(results) == top_n:
            break

    recommendations = df.iloc[results].copy()

    recommendations["similarity_score"] = np.round(
        similarities[results],
        4
    )

    recommendations = recommendations[[
        'name',
        'brand',
        'city',
        'price',
        'model',
        'kilometer',
        'fuel_type',
        'engine',
        'transmission',
        'similarity_score'
    ]]

    print("\n✅ RECOMMENDATIONS\n")

    print(recommendations.to_string(index=False))

    return recommendations


# =========================================================
# MAIN
# =========================================================

if __name__ == "__main__":


    # =====================================================
    # AFTER TRAINING
    # =====================================================

    recommend_cars(
        user_input="honda 2025",
        top_n=50
    )




📦 LOADING SAVED ARTIFACTS
✅ Artifacts Loaded

🔍 MATCHED:
Honda Civic  2025 RS
Score: 100.0
                     name     price  kilometer  engine  model fuel_type  \
200  Honda Civic  2025 RS  101000.0      11298    1500   2025    petrol   

    transmission  brand      city  \
200    automatic  honda  peshawar   

                                         search_text  
200  honda civic 2025 rs honda petrol automatic 2025  

✅ RECOMMENDATIONS

                                                              name brand     city     price  model  kilometer fuel_type  engine transmission  similarity_score
                               Honda Civic 11th Generation 2025 RS honda peshawar  102000.0   2025       9000    petrol    1500    automatic            1.0000
                               Honda Civic 11th Generation 2025 RS honda peshawar  105000.0   2025       5000    petrol    1500    automatic            1.0000
                               Honda Civic 11th Generation 2025 RS honda pe